# ADE Classifier — Analysis

Walkthrough of the final ablation results. Each figure and table is generated from `results/` via the `pv_ade.analysis` module — no retraining needed, runs locally in a few seconds.

## Context

Three transformer encoders fine-tuned on ADE Corpus v2 ([Gurulingappa et al., 2012](https://doi.org/10.1016/j.jbi.2012.04.008)) for binary sentence classification (ADE-related vs. not):

- **base BERT** — general-domain pretraining ([Devlin et al., 2019](https://arxiv.org/abs/1810.04805)). Baseline.
- **BioBERT** — continued pretraining on PubMed abstracts and PMC full-text articles, initialized from base BERT weights ([Lee et al., 2020](https://arxiv.org/abs/1901.08746)).
- **PubMedBERT** — pretrained from scratch on PubMed abstracts; the authors argue from-scratch domain pretraining outperforms continued pretraining for biomedical NLP ([Gu et al., 2021](https://arxiv.org/abs/2007.15779)).

Each model was fine-tuned with **ten random seeds** at its own LR-sweep winner — a per-model 3-LR sweep on val (seed 13) selected the learning rate for each backbone separately, so weaker backbones aren't penalized by another model's hyperparameters. Pairwise comparisons use paired bootstrap on pooled test predictions, following the significance-testing approach from [Koehn (2004)](https://aclanthology.org/W04-3250/).

## 1. Setup

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from sklearn.metrics import average_precision_score, precision_recall_curve

PROJECT_ROOT = Path.cwd().parent  # notebook lives in notebooks/
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from pv_ade.analysis import (
    load_run_results,
    pairwise_gaps,
    per_model_val_thresholds,
    per_seed_metrics_at_threshold,
    pool_predictions,
)

METRICS_DIR = PROJECT_ROOT / "results" / "metrics"
PREDICTIONS_DIR = PROJECT_ROOT / "results" / "predictions"
SWEEP_PREDICTIONS_DIR = PROJECT_ROOT / "results" / "sweep" / "predictions"
FIGURES_DIR = PROJECT_ROOT / "figures"
FIGURES_DIR.mkdir(exist_ok=True)

# Consistent ordering and colors for every figure and table.
MODEL_ORDER = ["bert-base", "biobert", "pubmedbert"]
MODEL_COLORS = {
    "bert-base": "#1f77b4",    # tab:blue  — baseline
    "biobert": "#ff7f0e",      # tab:orange — continued-pretraining variant
    "pubmedbert": "#2ca02c",   # tab:green  — from-scratch variant
}

BOOTSTRAP_SEED = 42
BOOTSTRAP_N_ITER = 1000

# Load per-model winner LRs and the sweep seed from the single source of truth.
with open(PROJECT_ROOT / "configs" / "ablation.yaml") as f:
    CFG = yaml.safe_load(f)
WINNER_LR_BY_MODEL = CFG["training"]["learning_rate_per_model"]
SWEEP_SEED = CFG["training"]["seeds"][0]  # sweep used the first seed
print("Winner LRs:", WINNER_LR_BY_MODEL)
print("Sweep seed:", SWEEP_SEED)

## 2. Load run results

One row per (model, seed). 30 rows total — 3 models × 10 seeds, each evaluated on the held-out test split at argmax threshold 0.5. These are the per-run metric values Colab wrote out.

In [ ]:
results = load_run_results(METRICS_DIR)
N_SEEDS = len(CFG["training"]["seeds"])
assert len(results) == len(MODEL_ORDER) * N_SEEDS, f"expected {len(MODEL_ORDER) * N_SEEDS} rows, got {len(results)}"
assert set(results["model"]) == set(MODEL_ORDER)
assert (results["eval_split"] == "test").all()

results["model"] = pd.Categorical(results["model"], categories=MODEL_ORDER, ordered=True)
results = results.sort_values(["model", "seed"]).reset_index(drop=True)
results[["model", "seed", "accuracy", "macro_f1", "f1_pos", "pr_auc"]]

## 3. Per-model summary (argmax threshold 0.5)

Mean ± std across seeds at the default 0.5 threshold — what Colab wrote. This captures training-run noise but says nothing about test-set sampling noise.

In [ ]:
summary_rows = []
for model in MODEL_ORDER:
    subset = results[results["model"] == model]
    row = {"model": model, "n_seeds": len(subset)}
    for metric in ("macro_f1", "f1_pos", "pr_auc"):
        m = subset[metric].mean()
        sd = subset[metric].std()
        row[f"{metric} (mean ± std)"] = f"{m:.4f} ± {sd:.4f}"
    summary_rows.append(row)

summary = pd.DataFrame(summary_rows)
summary

## 4. Threshold tuning on val

Argmax (0.5) is an arbitrary decision boundary. ADE Corpus v2 is imbalanced (positive rate ~0.20), so a per-model threshold tuned on val can shift macro-F1. This is a light sensitivity check, not the headline: the tuned thresholds are picked from a **single val run** (sweep seed 13) rather than cross-validated, so treat them as a cheap illustration of how much the 0.5 choice matters.

We sweep thresholds in `[0.05, 0.95]` at step 0.01 on each model's sweep-winner val predictions, picking the threshold that maximizes macro-F1. Ties near the top are broken toward 0.5 so the tuned value stays close to argmax when the metric is flat.

In [ ]:
thresholds = per_model_val_thresholds(
    SWEEP_PREDICTIONS_DIR, WINNER_LR_BY_MODEL, sweep_seed=SWEEP_SEED,
)
pd.DataFrame(
    [{"model": m, "winner_lr": WINNER_LR_BY_MODEL[m], "tuned_threshold": thresholds[m]} for m in MODEL_ORDER]
)

The tuned thresholds are `bert-base` 0.27, `biobert` 0.40, `pubmedbert` 0.94. `pubmedbert` is already very confident on its positives, so the optimal val cutpoint sits high — a useful datum in the sensitivity analysis below, because that high cutpoint interacts badly with pooled (seed-averaged) probabilities.

## 5. Per-seed summary at tuned thresholds

Recompute all classification metrics on each (model, seed) test NPZ using its tuned threshold. `pr_auc` is threshold-independent and is reported from the same probabilities for consistency.

In [ ]:
per_seed = per_seed_metrics_at_threshold(PREDICTIONS_DIR, thresholds)

tuned_summary_rows = []
for model in MODEL_ORDER:
    subset = per_seed[per_seed["model"] == model]
    row = {"model": model, "threshold": thresholds[model], "n_seeds": len(subset)}
    for metric in ("macro_f1", "f1_pos", "pr_auc"):
        m = subset[metric].mean()
        sd = subset[metric].std()
        row[f"{metric} (mean ± std)"] = f"{m:.4f} ± {sd:.4f}"
    tuned_summary_rows.append(row)

tuned_summary = pd.DataFrame(tuned_summary_rows)
tuned_summary

Per-seed means at tuned thresholds preserve the ranking: `pubmedbert` (macro_f1 ≈ 0.9364) > `biobert` (≈ 0.9328) > `bert-base` (≈ 0.9205). Threshold tuning buys `bert-base` essentially nothing here — the argmax was already close to its val optimum.

## 6. Figure 1 — Seed variance at argmax 0.5

Per-seed macro-F1 for each model with the cross-seed mean overlaid. The spread within each model is training-run noise; the spread between clusters is the between-model effect we're trying to measure. Using argmax 0.5 here so the figure matches the per-run JSON metrics without any tuning lens.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4), dpi=120)
rng = np.random.default_rng(0)

for i, model in enumerate(MODEL_ORDER):
    subset = results[results["model"] == model]
    x_jit = i + rng.uniform(-0.08, 0.08, size=len(subset))
    ax.scatter(
        x_jit, subset["macro_f1"],
        color=MODEL_COLORS[model], s=42, alpha=0.8,
        edgecolor="white", linewidth=0.7, zorder=3,
    )
    mean_val = subset["macro_f1"].mean()
    ax.hlines(
        mean_val, i - 0.2, i + 0.2,
        colors=MODEL_COLORS[model], linewidth=2.4, zorder=2,
    )

ax.set_xticks(range(len(MODEL_ORDER)))
ax.set_xticklabels(MODEL_ORDER)
ax.set_ylabel("test macro-F1 (threshold=0.5)")
ax.set_title(f"Per-seed macro-F1 with cross-seed mean (n={N_SEEDS} seeds per model)")
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()

fig.savefig(FIGURES_DIR / "seed_variance_macro_f1.png", dpi=200, bbox_inches="tight")
plt.show()

### Observations

- **Cluster ordering is consistent.** `pubmedbert` sits above `biobert` sits above `bert-base` — no seed of `bert-base` outperforms any seed of `pubmedbert`.
- **Seed noise is small but not negligible.** Per-model std ranges from ≈0.0035 (`pubmedbert`) to ≈0.0067 (`biobert`). That matters because the `biobert` vs `pubmedbert` gap (≈0.005 in means) is right at the noise floor.
- **`biobert` is noisier than the others.** Twice the spread of `pubmedbert` with a similar mean — worth a note if downstream users need deployment-stable calibration at a fixed threshold.

## 7. Pairwise gaps — macro-F1 (argmax threshold 0.5)

Paired bootstrap on pooled (seed-averaged) test predictions with 1000 iterations and a 95% CI. Pairs are emitted in alphabetical `model_a < model_b` order; the reported gap is `metric(a) − metric(b)`. CIs that exclude zero are statistically distinguishable differences.

Pooling rather than picking one seed focuses the bootstrap on test-set sampling noise; seed noise is captured separately by the per-model summary above.

In [ ]:
gaps_f1 = pairwise_gaps(
    results, PREDICTIONS_DIR,
    metric="macro_f1",
    n_iter=BOOTSTRAP_N_ITER,
    ci=0.95,
    seed=BOOTSTRAP_SEED,
)
gaps_f1["crosses_zero"] = (gaps_f1["ci_lo"] <= 0) & (gaps_f1["ci_hi"] >= 0)
gaps_f1

## 8. Pairwise gaps — PR-AUC

Same bootstrap machinery, but on pooled positive-class probabilities rather than thresholded labels. PR-AUC is a ranking metric, so it's sensitive to a model's probability ordering across the full test set — not just whether its threshold-based decisions are right. Threshold-independent.

In [ ]:
gaps_prauc = pairwise_gaps(
    results, PREDICTIONS_DIR,
    metric="pr_auc",
    n_iter=BOOTSTRAP_N_ITER,
    ci=0.95,
    seed=BOOTSTRAP_SEED,
)
gaps_prauc["crosses_zero"] = (gaps_prauc["ci_lo"] <= 0) & (gaps_prauc["ci_hi"] >= 0)
gaps_prauc

## 9. Figure 2 — Forest plot of pairwise gaps

Six horizontal error bars, three pairs × two metrics. Dashed line at zero. Gray markers cross zero (not distinguishable); red markers exclude zero (the statistically detectable differences).

In [ ]:
rows = []
for _, r in gaps_f1.iterrows():
    rows.append((
        "macro-F1",
        f"{r['model_a']} - {r['model_b']}",
        r["gap_macro_f1"], r["ci_lo"], r["ci_hi"], r["crosses_zero"],
    ))
for _, r in gaps_prauc.iterrows():
    rows.append((
        "PR-AUC",
        f"{r['model_a']} - {r['model_b']}",
        r["gap_pr_auc"], r["ci_lo"], r["ci_hi"], r["crosses_zero"],
    ))

fig, ax = plt.subplots(figsize=(7.5, 5), dpi=120)
y_positions = np.arange(len(rows))[::-1]
for y, (metric, label, point, lo, hi, crosses) in zip(y_positions, rows):
    color = "#888888" if crosses else "#c0392b"
    ax.errorbar(
        point, y,
        xerr=[[point - lo], [hi - point]],
        fmt="o", color=color,
        markersize=6, capsize=4, linewidth=1.6,
    )

ax.axvline(0, color="black", linestyle="--", alpha=0.5, linewidth=1)
ax.set_yticks(y_positions)
ax.set_yticklabels([f"{m}: {label}" for m, label, *_ in rows], fontsize=9)
ax.set_xlabel("gap (model_a − model_b)")
ax.set_title("Pairwise model gaps with 95% bootstrap CIs (threshold=0.5)")
ax.grid(axis="x", alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()

fig.savefig(FIGURES_DIR / "pairwise_gaps_forest.png", dpi=200, bbox_inches="tight")
plt.show()

### Observations

Four of the six CIs exclude zero — more signal than the 5-seed first pass surfaced:

- **`bert-base` − `pubmedbert` = −0.0121, 95% CI [−0.0236, −0.0025] (macro-F1).** PubMedBERT's from-scratch biomedical pretraining is a distinguishable win on thresholded decisions.
- **`biobert` − `pubmedbert` = −0.0085, 95% CI [−0.0176, −0.0002] (macro-F1).** From-scratch beats continued biomedical pretraining, but the upper bound sits right on the zero line — a ~2σ effect, not a landslide.
- **`bert-base` − `biobert` = −0.0036, CI [−0.0141, +0.0070] (macro-F1).** Not distinguishable from zero — at threshold 0.5, continued biomedical pretraining doesn't clearly beat general-domain pretraining.
- **PR-AUC tells a compatible ranking story.** Both biomedical variants beat `bert-base` on probability ordering (`bert-base` − `biobert` = −0.0117 CI [−0.0202, −0.0051]; `bert-base` − `pubmedbert` = −0.0177 CI [−0.0311, −0.0067]). The `biobert` vs `pubmedbert` PR-AUC gap is indistinguishable (−0.0060 CI [−0.0165, +0.0026]) — their probability orderings are about equally good; what separates them is where the 0.5 cut lands.

## 10. Figure 3 — PR curves on pooled predictions

Precision-recall trace for each model's seed-averaged positive-class probability, with the dataset's positive rate as a dashed floor (no-skill baseline). If the curves trace one another closely, that's visual evidence that PR-AUC can't separate the models.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5), dpi=120)

pooled_probs = {}
y_true_ref = None
for model in MODEL_ORDER:
    y_true, _, prob = pool_predictions(PREDICTIONS_DIR, model)
    if y_true_ref is None:
        y_true_ref = y_true
    pooled_probs[model] = prob

pos_rate = float(y_true_ref.mean())

for model in MODEL_ORDER:
    precision, recall, _ = precision_recall_curve(y_true_ref, pooled_probs[model])
    ap = average_precision_score(y_true_ref, pooled_probs[model])
    ax.plot(
        recall, precision,
        color=MODEL_COLORS[model],
        label=f"{model} (AP={ap:.4f})",
        linewidth=2, alpha=0.9,
    )

ax.axhline(
    pos_rate, color="gray", linestyle=":", linewidth=1,
    label=f"positive rate ({pos_rate:.3f})",
)

ax.set_xlabel("recall")
ax.set_ylabel("precision")
ax.set_title("PR curves on pooled (seed-averaged) test predictions")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.02)
ax.legend(loc="lower left", fontsize=9, framealpha=0.95)
ax.grid(alpha=0.3)
ax.set_axisbelow(True)
plt.tight_layout()

fig.savefig(FIGURES_DIR / "pr_curves.png", dpi=200, bbox_inches="tight")
plt.show()

### Observations

- **Biomedical models' curves sit above `bert-base`'s across most of the recall range.** AP values: `pubmedbert` ≈ 0.961, `biobert` ≈ 0.955, `bert-base` ≈ 0.943. The AP gap between biomedical variants is small (~0.006) and matches the indistinguishable bootstrap CI.
- **All three sit far above the no-skill baseline** (positive rate ≈ 0.20), so any of the three is a usable screen.

## 11. Sensitivity — threshold tuning on the pairwise gap

How much does the ranking depend on the 0.5 choice? Re-run the macro-F1 paired bootstrap with the per-model tuned thresholds passed through. Two caveats up front: thresholds were picked on a single val run (seed 13), and they're being applied to pooled (seed-averaged) probabilities — pooling smooths each seed's distribution, so a cutpoint tuned on one seed's curve can land in a different place on the pooled one. That's why pooled+tuned can disagree with the per-seed summary in §5. Reported side-by-side with the argmax gaps so the reader can see how much threshold choice moves the picture; the argmax bootstrap is still the headline.

In [ ]:
gaps_f1_tuned = pairwise_gaps(
    results, PREDICTIONS_DIR,
    metric="macro_f1",
    n_iter=BOOTSTRAP_N_ITER,
    ci=0.95,
    seed=BOOTSTRAP_SEED,
    thresholds_by_model=thresholds,
)
gaps_f1_tuned["crosses_zero"] = (gaps_f1_tuned["ci_lo"] <= 0) & (gaps_f1_tuned["ci_hi"] >= 0)

comparison = gaps_f1.merge(
    gaps_f1_tuned, on=["model_a", "model_b"], suffixes=("_argmax", "_tuned")
)[[
    "model_a", "model_b",
    "gap_macro_f1_argmax", "ci_lo_argmax", "ci_hi_argmax", "crosses_zero_argmax",
    "gap_macro_f1_tuned", "ci_lo_tuned", "ci_hi_tuned", "crosses_zero_tuned",
]]
comparison

### Observations

Applying val-tuned thresholds on pooled predictions reshuffles which pairs are distinguishable. `bert-base` − `biobert` becomes distinguishable (it wasn't at 0.5) because lowering `bert-base`'s threshold to 0.27 stretches its pooled decision boundary past `biobert`'s; `bert-base` − `pubmedbert` becomes indistinguishable (it wasn't at 0.5) because `pubmedbert`'s 0.94 threshold on a seed-averaged distribution throws away too many true positives. The per-seed summary in §5 did not exhibit this flip — the tuned thresholds were fit per-seed and evaluated per-seed, so no pooling/threshold interaction. Take this as evidence that pooled-bootstrap + tuned-threshold is a sensitivity check, not the headline, and that per-seed evaluation is the more faithful view when operating at a tuned cutpoint.

## 12. Findings

### Signal vs noise

Per-seed macro-F1 std sits in the low single-digit thousandths (≈0.003–0.007). Between-model means span a similar order (`bert-base` 0.9200, `biobert` 0.9327, `pubmedbert` 0.9377 at argmax 0.5). Four of the six pairwise bootstrap CIs exclude zero — more signal than the 5-seed first pass surfaced, consistent with `n=10` seeds tightening the variance on pooled predictions.

### Main picture (argmax 0.5, pooled bootstrap)

- `pubmedbert` beats `bert-base` on both macro-F1 (−0.0121 CI [−0.0236, −0.0025]) and PR-AUC (−0.0177 CI [−0.0311, −0.0067]).
- `pubmedbert` beats `biobert` on macro-F1 by a thinner margin that still excludes zero (−0.0085 CI [−0.0176, −0.0002]); PR-AUC between them is indistinguishable.
- `biobert` beats `bert-base` on PR-AUC (−0.0117 CI [−0.0202, −0.0051]) but not on macro-F1 (−0.0036 CI [−0.0141, +0.0070]). Biomedical pretraining cleans up `bert-base`'s probability ordering but doesn't reliably improve its thresholded decisions at 0.5.

### Interpretation

The pattern is internally consistent with "biomedical pretraining helps probability ranking; from-scratch biomedical pretraining additionally helps decision quality at a standard cutpoint." `biobert` picks up the ranking gain over `bert-base`; `pubmedbert` picks up both the ranking gain and the threshold gain. This is the direction the Gu et al. (2021) paper argues for — though the effect sizes here are modest (~1pp macro-F1) and the task is narrow.

### Threshold tuning is a sensitivity check, not a headline

Per-seed means with tuned thresholds preserve the ranking and barely move the numbers. Pooled bootstrap with tuned thresholds (§11) reshuffles which pairs are distinguishable — an artifact of the seed-averaged distribution interacting with val-tuned cutpoints, not a real effect. The argmax-0.5 bootstrap is the defensible headline.

### What changed from the 5-seed pilot

The earlier 5-seed run used one shared LR (5e-5) across all three backbones. The per-model sweep pinned `bert-base` at 1e-4 and the two biomedical models at 5e-5. That change alone shifts `bert-base`'s per-seed macro-F1 downward by a few points on test — a reminder that single-config ablations can silently favor whichever model the shared config was tuned on. The 10-seed + per-model-LR replication flipped the headline effect (from "base BERT beats BioBERT" at n=5 with shared LR, to "biomedical beats base BERT and from-scratch beats continued" at n=10 with per-model LR).

### Scope limits

- Single test split (~2.1k sentences) from a committed stratified 80/10/10.
- Per-model LR picked from a sweep at one seed on val; thresholds tuned on the same one-seed val run.
- Sentence-level binary classification — no claim transfers to entity extraction, drug–event relation linking, or clinical document tasks.
- Ten seeds per model is a reasonable floor for a small ablation; a larger replication would tighten every CI.

## References

- Devlin, J., Chang, M.-W., Lee, K., & Toutanova, K. (2019). BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding. *NAACL-HLT 2019*. [arXiv:1810.04805](https://arxiv.org/abs/1810.04805)
- Gu, Y., Tinn, R., Cheng, H., Lucas, M., Usuyama, N., Liu, X., Naumann, T., Gao, J., & Poon, H. (2021). Domain-Specific Language Model Pretraining for Biomedical Natural Language Processing. *ACM Transactions on Computing for Healthcare, 3(1)*. [arXiv:2007.15779](https://arxiv.org/abs/2007.15779)
- Gurulingappa, H., Rajput, A. M., Roberts, A., Fluck, J., Hofmann-Apitius, M., & Toldo, L. (2012). Development of a benchmark corpus to support the automatic extraction of drug-related adverse effects from medical case reports. *Journal of Biomedical Informatics, 45*(5), 885–892. [doi:10.1016/j.jbi.2012.04.008](https://doi.org/10.1016/j.jbi.2012.04.008)
- Koehn, P. (2004). Statistical Significance Tests for Machine Translation Evaluation. *EMNLP 2004*, 388–395. [ACL Anthology W04-3250](https://aclanthology.org/W04-3250/)
- Lee, J., Yoon, W., Kim, S., Kim, D., Kim, S., So, C. H., & Kang, J. (2020). BioBERT: a pre-trained biomedical language representation model for biomedical text mining. *Bioinformatics, 36*(4), 1234–1240. [arXiv:1901.08746](https://arxiv.org/abs/1901.08746)